In [2]:
//#r "C:\Aktuell_BoSSS_2\experimental\public\src\L4-application\BoSSSpad\bin\Debug\net6.0\BoSSSpad.dll"
#r "C:\Users\flori\Documents\BoSSS-premaster\public\src\L4-application\BoSSSpad\bin\Debug\net6.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Platform.LinAlg;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using BoSSS.Application.XNSFE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();


In [106]:
var grd = Grid2D.Cartesian2DGrid(GenericBlas.Linspace(-1, 1, 5), GenericBlas.Linspace(-1, 1, 5));

In [107]:
Basis b = new Basis(grd, 2);

In [108]:
SinglePhaseField u0 = new SinglePhaseField(b, "u0");
u0.ProjectField((x, y) => (1-x).Pow2()*(1-y).Pow2());

In [109]:
u0.CoordinateVector

PresentExternal,False
Mapping,[ u0 ]
Fields,[ u0 ]
Count,96
Length,96
IsReadOnly,False
(values),"[ 4.753472222222224, -0.77882145687559, -0.7788214568755898, 0.028727262210938187, 0.1276041666666671, 0.028727262210938187, 2.4409722222222223, -0.5563010406254211, -0.3999353427198975, 0.028727262210936556, 0.09114583333333345, 0.014751837351562797, 0.8993055555555556, -0.3337806243752525, -0.14734459994943586, 0.02872726221093545, 0.05468750000000014, 0.005434887445312633, 0.12847222222222224, -0.1112602081250842, -0.021049228564205127, 0.028727262210934957, 0.0182291666666667, 0.0007764124921875172, 2.4409722222222223, -0.3999353427198973, -0.5563010406254213, 0.014751837351562859, 0.09114583333333351, 0.028727262210936612, 1.2534722222222219, -0.28566810194278386, -0.2856681019427837, 0.014751837351562078, 0.06510416666666677, 0.014751837351562023, 0.4618055555555555, -0.17140086116567024, -0.10524614282102565, 0.014751837351561502, 0.03906250000000004, 0.0054348874453123116, 0.06597222222222222, -0.05713362038855673, -0.015035163260146518, 0.0147518373515612, 0.01302083333333335, 0.0007764124921874757, 0.8993055555555556, -0.14734459994943588, -0.33378062437525274, 0.005434887445312609, 0.054687500000000125, 0.028727262210935495, 0.4618055555555555, -0.10524614282102562, -0.17140086116567024, 0.0054348874453123185, 0.03906250000000007, 0.014751837351561478, 0.17013888888888892, -0.06314768569261535, -0.06314768569261535, 0.005434887445312138, 0.02343750000000003, 0.005434887445312132, 0.024305555555555556, -0.021049228564205123, -0.009021097956087906, 0.005434887445312023, 0.007812500000000007, 0.0007764124921874463, 0.12847222222222224, -0.02104922856420512, -0.11126020812508419, 0.000776412492187508, 0.018229166666666706, 0.028727262210934967, 0.06597222222222224, -0.015035163260146525, -0.05713362038855674, 0.0007764124921874733, 0.013020833333333355, 0.0147518373515612, 0.024305555555555556, -0.009021097956087907, -0.021049228564205116, 0.000776412492187446, 0.007812500000000012, 0.005434887445312021, 0.003472222222222223, -0.0030070326520293027, -0.003007032652029303, 0.000776412492187432, 0.002604166666666669, 0.0007764124921874317 ]"


In [110]:
u0.CoordinateVector.L2Norm()

6.399976226385728

In [111]:
static Vector VelocityField(Vector X) {
    if(X.Dim != 2)
        throw new ArgumentException("expecting 2D input vector");
    Vector U = new Vector(X.y, -X.x);
    return U;
}

In [112]:
VelocityField(new Vector(0.8, 0.2))

Count,2
IsReadOnly,True
x,0.2
y,-0.8
z,0
Dim,2
Dummy_256bitAlign,0
(values),"[ 0.2, -0.8 ]"


In [113]:
static double UpwindFlux(Vector X, Vector N, double uIn, double uOt) {
    var Vel = VelocityField(X);
    if(Vel*N > 0) {
        return (Vel*N)*uIn;
    } else {
        return (Vel*N)*uOt;
    }
}

In [114]:
class WeakForm :  IEdgeForm, IVolumeForm {

    public IList<String> ArgumentOrdering {
        get { return new string[] { "u" }; }
    }

    public IList<String> ParameterOrdering => null;

    public TermActivationFlags VolTerms => TermActivationFlags.AllOn; 

    public TermActivationFlags InnerEdgeTerms => TermActivationFlags.AllOn; 

    public TermActivationFlags BoundaryEdgeTerms => TermActivationFlags.AllOn; 
           
    public double VolumeForm(ref CommonParamsVol cpv, double[] U, double[,] GradU, double V, double[] GradV) {
        double u = U[0];
        var Vel = VelocityField(cpv.Xglobal);
        return -((Vel*u)*GradV);
    }

    public double InnerEdgeForm(ref CommonParams inp, double[] Uin, double[] Uout, double[,] _Grad_uIN, double[,] _Grad_uOUT, double V_IN, double V_OT, double[] _Grad_vIN, double[] _Grad_vOUT) {
        //return 0;
        return UpwindFlux(inp.X, inp.Normal, Uin[0], Uout[0])*(V_IN - V_OT);
    }

    public double BoundaryEdgeForm(ref CommonParamsBnd inp, double[] Uin, double[,] GradUin, double vIn, double[] GradVin) {
        // V1 ok;
        //var Vel = VelocityField(inp.X);
        //var uIn = Uin[0];
        //return ((Vel*uIn)*inp.Normal)*vIn;

        //V2
        return UpwindFlux(inp.X, inp.Normal, Uin[0], 0.0)*vIn;
    }
}

In [115]:
var OperatorWeak = (new WeakForm()).Operator();

In [116]:
SinglePhaseField WeakFlux = new SinglePhaseField(b);
OperatorWeak.Evaluate(u0, WeakFlux);
WeakFlux.L2Norm()

31.164776966098753

In [117]:
class SuperWeakForm :  IEdgeForm, IVolumeForm {

    public IList<String> ArgumentOrdering {
        get { return new string[] { "u" }; }
    }

    public IList<String> ParameterOrdering => null;

    public TermActivationFlags VolTerms => TermActivationFlags.AllOn; 

    public TermActivationFlags InnerEdgeTerms => TermActivationFlags.AllOn; 

    public TermActivationFlags BoundaryEdgeTerms => TermActivationFlags.AllOn; 
           
    public double VolumeForm(ref CommonParamsVol cpv, double[] U, double[,] GradU, double v, double[] GradV) {
        // equal to div(Vel*u) for div(Vel) = 0
        double u = U[0];
        var Vel = VelocityField(cpv.Xglobal);
        var Grad_u = GradU.GetRowPt(0);
        return (Vel*Grad_u)*v;
    }

    public double InnerEdgeForm(ref CommonParams inp, double[] Uin, double[] Uout, double[,] _Grad_uIN, double[,] _Grad_uOUT, double V_IN, double V_OT, double[] _Grad_vIN, double[] _Grad_vOUT) {
        var Vel = VelocityField(inp.X);
        var uIn = Uin[0];
        var uOt = Uout[0];

        //return -(Vel*inp.Normal*uIn*V_IN + Vel*(-1.0*inp.Normal)*uOt*V_OT);
        double Flux = UpwindFlux(inp.X, inp.Normal, Uin[0], Uout[0]);
        return (Flux - (Vel*uIn)*inp.Normal)*V_IN - (Flux - (Vel*uOt)*inp.Normal)*V_OT;
    }

    public double BoundaryEdgeForm(ref CommonParamsBnd inp, double[] Uin, double[,] GradUin, double vIn, double[] GradVin) {
        var Vel = VelocityField(inp.X);
        var uIn = Uin[0];
        
        // V1 ok
        //return 0;
        
        // V2 ok
        return (UpwindFlux(inp.X, inp.Normal, Uin[0], 0.0) - (Vel*uIn)*inp.Normal )*vIn;
    }
}

In [118]:
var OperatorSuperWeak = (new SuperWeakForm()).Operator();

In [119]:
SinglePhaseField SuperWeakFlux = new SinglePhaseField(b);
OperatorSuperWeak.Evaluate(u0, SuperWeakFlux);
SuperWeakFlux.L2Norm()

31.164776966098753

In [120]:
WeakFlux.CoordinateVector.L2Distance(SuperWeakFlux.CoordinateVector)

2.394481992412156E-14

In [121]:
WeakFlux.CoordinateVector.ToArray()

[ 9.538628472222246, -17.63308911986723, -2.1792466801365196, 21.46411744964452, 5.687586805555564, 0.18711541061718506, -2.197916666666673, -0.22763237175859619, 0.9763835021139097, 0.08851102410935408, -0.04374999999999951, -0.10248644896873904, -1.5299479166666698, 0.4111114807432834, -0.15471182994690558, -0.04736116202343954, -0.2813368055555576, 0.9648866246659275, -0.47873263888889045, 0.2487317175336941, -0.14539002872561646, 0.022515962273432576, 0.21032986111111138, 0.4435256361620684, 3.9874131944444535, -4.074679595132306, -1.0546164682775565, 4.096934618150012, 2.241753472222224, 0.11723828632030475, -8.326672684688674E-16, -0.34370383212694544, 0.3449066451877596, 0.06288941186717956, -0.010416666666668406, -0.065218649343743, -0.47916666666666763, 0.015937273055756228, 0.26251395052215964, 0.027950849718746992, -0.018749999999999475, -0.03959703710155943, -0.1770833333333338, 0.10614825261663506, 0.08991027629567688, -0.009316949906249768, -0.02708333333333354, -0.013975424859373867, 1.9895833333333375, -0.6401972516170404, -0.2775491137823094, 0.055901699437494345, 0.05625000000000019, -0.041926274578116665, 0.4791666666666675, -0.2613111374613473, -0.017140086116567588, 0.03726779962499635, 0.022916666666666807, -0.03028008719530928, 2.7755575615628914E-17, -0.046007599576048656, 0.04480478651523728, 0.016304662335936207, 0.006250000000000429, -0.01863389981249841, -0.009114583333333356, 0.010374262649500922, 0.01899442291865208, -0.004076165583983925, -0.0019965277777775486, -0.005434887445312137, 0.6844618055555568, -0.19941638204041065, -0.40910679230858765, 0.014751837351560604, 0.09574652777777784, -0.0021351343535137243, 0.1749131944444448, -0.0851491412632968, -0.11110985649248283, 0.010093362398436695, 0.041579861111111144, -0.0025233405996088884, 0.010416666666666588, -0.017741492646973108, -0.009321801221290489, 0.0046584749531246306, 0.014583333333333628, -3.2786273695961654E-16, 0.00043402777777775715, 0.0010524614282102114, 0.000952227006476054, -0.003687959337890262, 0.006336805555555599, -0.0007764124921875142 ]

In [122]:
SuperWeakFlux.CoordinateVector.ToArray()

[ 9.538628472222248, -17.633089119867222, -2.1792466801365347, 21.46411744964452, 5.687586805555563, 0.1871154106171746, -2.197916666666674, -0.2276323717585914, 0.9763835021139091, 0.0885110241093597, -0.04374999999999999, -0.10248644896873949, -1.529947916666671, 0.41111148074328535, -0.15471182994690458, -0.047361162023437355, -0.2813368055555581, 0.9648866246659273, -0.4787326388888907, 0.24873171753369436, -0.14539002872561613, 0.02251596227343354, 0.21032986111111104, 0.4435256361620686, 3.9874131944444544, -4.074679595132305, -1.054616468277564, 4.096934618150013, 2.241753472222226, 0.1172382863203003, -6.314527304675637E-16, -0.343703832126945, 0.34490664518775865, 0.06288941186718061, -0.010416666666667794, -0.06521864934374386, -0.4791666666666678, 0.01593727305575659, 0.26251395052216003, 0.02795084971874751, -0.018749999999999788, -0.0395970371015592, -0.17708333333333381, 0.1061482526166351, 0.08991027629567702, -0.009316949906249527, -0.027083333333333647, -0.013975424859373795, 1.9895833333333381, -0.6401972516170416, -0.27754911378231123, 0.055901699437494165, 0.05625000000000057, -0.041926274578120995, 0.47916666666666774, -0.26131113746134776, -0.017140086116568004, 0.03726779962499649, 0.02291666666666677, -0.030280087195309592, 4.907286425637476E-17, -0.04600759957604879, 0.04480478651523735, 0.016304662335936103, 0.006250000000000419, -0.018633899812498328, -0.009114583333333369, 0.010374262649500938, 0.018994422918652134, -0.004076165583983984, -0.001996527777777598, -0.00543488744531213, 0.6844618055555571, -0.19941638204041112, -0.40910679230858793, 0.014751837351560696, 0.09574652777777812, -0.00213513435351494, 0.1749131944444449, -0.08514914126329694, -0.1111098564924829, 0.010093362398436523, 0.04157986111111127, -0.0025233405996090558, 0.010416666666666585, -0.017741492646973167, -0.00932180122129051, 0.00465847495312455, 0.014583333333333639, -2.5934115965853266E-16, 0.00043402777777775856, 0.001052461428210211, 0.0009522270064760466, -0.003687959337890279, 0.0063368055555555955, -0.0007764124921874897 ]